🚀 [Run in JupyterLite](../lite/lab/index.html?path=lecture6.ipynb){target="_blank" .btn .btn-primary}

# Lecture 6: Data Manipulation with Pandas

# SARS-CoV-2 as a research problem

To learn about Pandas we will use SARS-CoV-2 data. Before actually jumping to Pandas let's learn about the coronavirus molecular biology.

The following summary is based on these publications:

- [Masters:2006](http://dx.doi.org/10.1016/S0065-3527(06)66005-3)
- [Fehr and Perlman:2015](http://dx.doi.org/10.1007/978-1-4939-2438-7_1)
- [Sola:2015](https://www.annualreviews.org/doi/full/10.1146/annurev-virology-100114-055218)
- [Kirchdoerfer:2016](http://dx.doi.org/10.1038/nature17200)
- [Walls:2020](http://dx.doi.org/10.1016/j.cell.2020.02.058)
- [Jackson:2022](https://www.nature.com/articles/s41580-021-00418-x)

## Genome organization

All coronaviruses contain non-segmented positive-strand RNA genome approx. 30 kb in length. It is invariably 5'-leader-UTR-replicase-S-E-M-N-3'UTR-poly(A). In addition, it contains a variety of accessory proteins interspersed throughout the genome.

![Genome organization](https://media.springernature.com/original/springer-static/image/chp%3A10.1007%2F978-1-4939-2438-7_1/MediaObjects/317916_1_En_1_Fig1_HTML.gif)

*Genomic organization of representative α, β, and γ CoVs. (From Fehr and Perlman:2015)*

## Virion structure

Coronavirus is a spherical particle approx. 125 nm in diameter. It is covered with S-protein projections giving it an appearance of solar corona - hence the term coronavirus. There are four main structure proteins: spike (S), membrane (M), envelope (E), and nucleocapsid (N).

![Virion structure](https://ars.els-cdn.com/content/image/1-s2.0-S0065352706660053-gr1.jpg)

*Schematic of the coronavirus virion (From Masters:2006)*

---

# Pandas!

> This is an aggregated tutorial relying on material from:
> - [Justin Bois](http://justinbois.github.io/bootcamp/2020/index.html)
> - [BIOS821 course at Duke](https://people.duke.edu/~ccc14/bios-821-2017/index.html)
> - [Pandas documentation](https://pandas.pydata.org/docs/user_guide/index.html/)

Pandas (from "Panel Data") is an essential piece of scientific (and not only) data analysis infrastructure. It is, in essence, a highly optimized library for manipulating very large tables (or "Data Frames").

## Pandas learning resources

- [Getting started](https://pandas.pydata.org/docs/getting_started/index.html#getting-started) - official introduction from Pandas.
- [Data Science Tools](http://people.duke.edu/~ccc14/bios-821-2017/index.html) - Data Science for Biologists from Duke University.
- [Data Carpentry](https://datacarpentry.org/) - a collection of lessons *à la* Software Carpentry.

In [1]:
# Pandas, conventionally imported as pd
import pandas as pd

Throughout your research career, you will undoubtedly need to handle data, possibly lots of data. The data comes in lots of formats, and you will spend much of your time **wrangling** the data to get it into a usable form.

Pandas is the primary tool in the Python ecosystem for handling data. Its primary object, the `DataFrame` is extremely useful in wrangling data.

# Basics

## The data set

The dataset we will be using is a subset of metadata describing SARS-CoV-2 datasets from the [Sequence Read Archive](https://www.ncbi.nlm.nih.gov/sra).

It is obtained by going to https://www.ncbi.nlm.nih.gov/sra and performing a query with the following search terms: `txid2697049[Organism:noexp]`.

::: {.callout-note}
## JupyterLite Compatibility
This notebook checks if data files exist locally before downloading. In JupyterLite, files are pre-loaded so no download is needed. In standard Jupyter, the files will be downloaded automatically. The shell equivalent would be:
```bash
!curl -sLO https://zenodo.org/records/10680001/files/sra_ncov.csv.gz
```
:::

In [2]:
import os
filename = "sra_ncov.csv.gz"
if not os.path.exists(filename):
    from urllib.request import urlretrieve
    urlretrieve("https://zenodo.org/records/10680001/files/sra_ncov.csv.gz", filename)
    print(f"Downloaded: {filename}")
else:
    print(f"Using pre-loaded file: {filename}")

Using pre-loaded file: sra_ncov.csv.gz


::: {.callout-note}
## JupyterLite Compatibility
The code below uses Python's `gzip` module to preview the compressed file. In a standard Jupyter environment, you could use:
```bash
!gunzip -c sra_ncov.csv.gz | head -3
```
:::

In [3]:
import gzip
with gzip.open('sra_ncov.csv.gz', 'rt') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(line.strip())

Run,ReleaseDate,size_MB,Experiment,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,Platform,Model,SRAStudy,BioProject
ERR5063394,2021-01-18 12:14:33,0,ERX4869505,Targeted-Capture,RANDOM,TRANSCRIPTOMIC,SINGLE,ILLUMINA,unspecified,ERP121228,PRJEB37886
ERR5063392,2021-01-18 12:14:33,0,ERX4869504,AMPLICON,PCR,VIRAL RNA,SINGLE,ILLUMINA,Illumina MiSeq,ERP121228,PRJEB37886


## Reading in data

Pandas has a very powerful function, `pd.read_csv()` that can read in a CSV file and store the contents in a convenient data structure called a **data frame**.

In [4]:
df = pd.read_csv('sra_ncov.csv.gz')

In [5]:
# View the first few rows
df.head()

,Run,ReleaseDate,size_MB,Experiment,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,Platform,Model,SRAStudy,BioProject
0,ERR5063394,2021-01-18 12:14:33,0,ERX4869505,Targeted-Capture,RANDOM,TRANSCRIPTOMIC,SINGLE,ILLUMINA,unspecified,ERP121228,PRJEB37886
1,ERR5063392,2021-01-18 12:14:33,0,ERX4869504,AMPLICON,PCR,VIRAL RNA,SINGLE,ILLUMINA,Illumina MiSeq,ERP121228,PRJEB37886
2,ERR5063395,2021-01-18 12:14:33,0,ERX4869506,Targeted-Capture,RANDOM,TRANSCRIPTOMIC,SINGLE,ILLUMINA,unspecified,ERP121228,PRJEB37886
3,ERR5063397,2021-01-18 12:14:33,0,ERX4869510,AMPLICON,PCR,VIRAL RNA,SINGLE,ILLUMINA,NextSeq 550,ERP121228,PRJEB37886
4,ERR5063396,2021-01-18 12:14:33,0,ERX4869507,AMPLICON,PCR,VIRAL RNA,SINGLE,ILLUMINA,NextSeq 550,ERP121228,PRJEB37886


## Indexing data frames

The data frame is a convenient data structure for many reasons. Let's start by looking at how data frames are indexed.

**Important**: We index DataFrames by columns, not rows!

In [6]:
# This gives us a column
df['Run'].head()

0    ERR5063394
1    ERR5063392
2    ERR5063395
3    ERR5063397
4    ERR5063396
Name: Run, dtype: object

In [7]:
# Access a single value
df['Run'][4]

'ERR5063396'

However, it's better to use `.loc` for accessing data. This gives the location in the data frame we want.

::: {.callout-tip}
## `loc` versus `iloc`

- `loc`: Label-based indexing - use actual row and column labels
- `iloc`: Integer-based indexing - use integer positions
:::

In [8]:
df.loc[4, 'Run']

'ERR5063396'

In [9]:
df.iloc[4:6]

,Run,ReleaseDate,size_MB,Experiment,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,Platform,Model,SRAStudy,BioProject
4,ERR5063396,2021-01-18 12:14:33,0,ERX4869507,AMPLICON,PCR,VIRAL RNA,SINGLE,ILLUMINA,NextSeq 550,ERP121228,PRJEB37886
5,ERR5063399,2021-01-18 12:14:33,0,ERX4869508,Targeted-Capture,RANDOM,TRANSCRIPTOMIC,SINGLE,ILLUMINA,unspecified,ERP121228,PRJEB37886


In [10]:
df.iloc[4:6, [0, 2, 4]]

,Run,size_MB,LibraryStrategy
4,ERR5063396,0,AMPLICON
5,ERR5063399,0,Targeted-Capture


In [11]:
df.loc[4:6, ['Run', 'size_MB', 'LibraryStrategy']]

,Run,size_MB,LibraryStrategy
4,ERR5063396,0,AMPLICON
5,ERR5063399,0,Targeted-Capture
6,ERR5063400,0,AMPLICON


## Filtering: Boolean indexing of data frames

Let's say I wanted to pull out accession numbers of runs produced by Pacific Biosciences machines (labeled as `PACBIO_SMRT`). I can use Boolean indexing to specify the row.

In [12]:
df.loc[df['Platform'] == 'PACBIO_SMRT', 'Run']

49441     SRR13144531
49442     SRR13144533
49443     SRR13144530
49444     SRR13144529
49445     SRR13144528
49446     SRR13144534
49447     SRR13144527
49448     SRR13144526
49449     SRR13144525
49450     SRR13144524
49451     SRR13144523
49452     SRR13144532
173270    SRR12038589
173271    SRR12038590
Name: Run, dtype: object

In [13]:
# Pull the whole record
df.loc[df['Platform'] == 'PACBIO_SMRT', :].head(10)

,Run,ReleaseDate,size_MB,Experiment,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,Platform,Model,SRAStudy,BioProject
49441,SRR13144531,2020-11-25 21:54:15,1137,SRX9584893,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710
49442,SRR13144533,2020-11-25 21:54:15,1999,SRX9584891,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710
49443,SRR13144530,2020-11-25 21:54:15,81,SRX9584894,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710
49444,SRR13144529,2020-11-25 21:54:15,1311,SRX9584895,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710
49445,SRR13144528,2020-11-25 21:54:15,327,SRX9584896,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710
49446,SRR13144534,2020-11-25 21:54:15,2292,SRX9584890,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710
49447,SRR13144527,2020-11-25 21:54:15,3377,SRX9584897,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710
49448,SRR13144526,2020-11-25 21:54:15,285,SRX9584898,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710
49449,SRR13144525,2020-11-25 21:54:15,406,SRX9584899,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710
49450,SRR13144524,2020-11-25 21:54:15,1507,SRX9584900,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710


Now, let's pull out all PacBio records that were obtained from Amplicon sequencing. We can use the `&` operator:

In [14]:
df.loc[(df['Platform'] == 'PACBIO_SMRT') & (df['LibraryStrategy'] == 'AMPLICON'), :].head(3)

,Run,ReleaseDate,size_MB,Experiment,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,Platform,Model,SRAStudy,BioProject
49441,SRR13144531,2020-11-25 21:54:15,1137,SRX9584893,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710
49442,SRR13144533,2020-11-25 21:54:15,1999,SRX9584891,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710
49443,SRR13144530,2020-11-25 21:54:15,81,SRX9584894,AMPLICON,RT-PCR,VIRAL RNA,SINGLE,PACBIO_SMRT,Sequel II,SRP294181,PRJNA680710


In [15]:
# See how many match
import numpy as np
inds = (df['Platform'] == 'PACBIO_SMRT') & (df['LibraryStrategy'] == 'AMPLICON')
np.unique(inds, return_counts=True)

(array([False,  True]), array([190344,     12]))

## Calculating with data frames

Let's add a column that specifies whether or not the corresponding run is above 100 MB:

In [16]:
# Add the column to the DataFrame
df['Over100Mb'] = df['size_MB'] >= 100

# Take a look
df.head()

,Run,ReleaseDate,size_MB,Experiment,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,Platform,Model,SRAStudy,BioProject,Over100Mb
0,ERR5063394,2021-01-18 12:14:33,0,ERX4869505,Targeted-Capture,RANDOM,TRANSCRIPTOMIC,SINGLE,ILLUMINA,unspecified,ERP121228,PRJEB37886,False
1,ERR5063392,2021-01-18 12:14:33,0,ERX4869504,AMPLICON,PCR,VIRAL RNA,SINGLE,ILLUMINA,Illumina MiSeq,ERP121228,PRJEB37886,False
2,ERR5063395,2021-01-18 12:14:33,0,ERX4869506,Targeted-Capture,RANDOM,TRANSCRIPTOMIC,SINGLE,ILLUMINA,unspecified,ERP121228,PRJEB37886,False
3,ERR5063397,2021-01-18 12:14:33,0,ERX4869510,AMPLICON,PCR,VIRAL RNA,SINGLE,ILLUMINA,NextSeq 550,ERP121228,PRJEB37886,False
4,ERR5063396,2021-01-18 12:14:33,0,ERX4869507,AMPLICON,PCR,VIRAL RNA,SINGLE,ILLUMINA,NextSeq 550,ERP121228,PRJEB37886,False


## A note about vectorization

Notice how applying the `>=` operator to a `Series` resulted in **elementwise** application. This is called **vectorization**. It means that we do not have to write a `for` loop to do operations on the elements of a `Series`.

Vectorized code is almost always faster because the looping is done with compiled code under the hood.

## Outputting a new CSV file

::: {.callout-note}
## JupyterLite Compatibility
The code below uses Python's built-in file reading to preview the file. In a standard Jupyter environment, you could use:
```bash
!head -3 over100Mb_data.csv
```
:::

In [17]:
df.to_csv('over100Mb_data.csv', index=False)

In [18]:
with open('over100Mb_data.csv', 'r') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(line.strip())

Run,ReleaseDate,size_MB,Experiment,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,Platform,Model,SRAStudy,BioProject,Over100Mb
ERR5063394,2021-01-18 12:14:33,0,ERX4869505,Targeted-Capture,RANDOM,TRANSCRIPTOMIC,SINGLE,ILLUMINA,unspecified,ERP121228,PRJEB37886,False
ERR5063392,2021-01-18 12:14:33,0,ERX4869504,AMPLICON,PCR,VIRAL RNA,SINGLE,ILLUMINA,Illumina MiSeq,ERP121228,PRJEB37886,False


---

# Tidy data

[Hadley Wickham](https://en.wikipedia.org/wiki/Hadley_Wickham) wrote a [great article](http://dx.doi.org/10.18637/jss.v059.i10) in favor of "tidy data." Tidy data frames follow the rules:

1. Each variable is a column.
2. Each observation is a row.
3. Each type of observation has its own separate data frame.

A tidy data frame is almost always **much** easier to work with than non-tidy formats.

## Finding unique values and counts

In [19]:
# Re-read the data (using local file downloaded earlier)
df = pd.read_csv('sra_ncov.csv.gz')
df = df[df['size_MB'] > 0].reset_index(drop=True)

In [20]:
df['Platform'].unique()

array(['ILLUMINA', 'OXFORD_NANOPORE', 'ION_TORRENT', 'PACBIO_SMRT',
       'BGISEQ'], dtype=object)

In [21]:
df['Platform'].value_counts()

Platform
ILLUMINA           155937
OXFORD_NANOPORE     25202
ION_TORRENT           507
BGISEQ                 22
PACBIO_SMRT            14
Name: count, dtype: int64

## Sorting

In [22]:
df_subset = df.sample(n=10)
df_subset

,Run,ReleaseDate,size_MB,Experiment,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,Platform,Model,SRAStudy,BioProject
82853,ERR4707958,2020-10-24 12:32:35,65,ERX4628384,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
107179,SRR12754167,2020-10-01 03:57:34,70,SRX9226122,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,NextSeq 550,SRP253798,PRJNA613958
91317,ERR4688199,2020-10-16 16:33:39,86,ERX4609501,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
80958,SRR12895638,2020-10-26 02:16:41,80,SRX9360797,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,NextSeq 550,SRP253798,PRJNA613958
50176,ERR4861036,2020-11-21 16:41:49,67,ERX4730419,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
17189,ERR5006023,2021-01-06 21:59:08,140,ERX4814955,AMPLICON,PCR,VIRAL RNA,SINGLE,ION_TORRENT,Ion Torrent PGM,ERP124690,PRJEB40971
32077,ERR4905843,2020-12-04 08:15:36,29,ERX4772667,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
42507,ERR4875827,2020-11-27 12:38:46,22,ERX4744689,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
72419,ERR4787607,2020-11-04 09:00:36,12,ERX4657365,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
82864,ERR4707953,2020-10-24 12:32:35,13,ERX4628379,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886


In [23]:
df_subset.sort_index()

,Run,ReleaseDate,size_MB,Experiment,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,Platform,Model,SRAStudy,BioProject
17189,ERR5006023,2021-01-06 21:59:08,140,ERX4814955,AMPLICON,PCR,VIRAL RNA,SINGLE,ION_TORRENT,Ion Torrent PGM,ERP124690,PRJEB40971
32077,ERR4905843,2020-12-04 08:15:36,29,ERX4772667,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
42507,ERR4875827,2020-11-27 12:38:46,22,ERX4744689,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
50176,ERR4861036,2020-11-21 16:41:49,67,ERX4730419,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
72419,ERR4787607,2020-11-04 09:00:36,12,ERX4657365,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
80958,SRR12895638,2020-10-26 02:16:41,80,SRX9360797,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,NextSeq 550,SRP253798,PRJNA613958
82853,ERR4707958,2020-10-24 12:32:35,65,ERX4628384,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
82864,ERR4707953,2020-10-24 12:32:35,13,ERX4628379,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
91317,ERR4688199,2020-10-16 16:33:39,86,ERX4609501,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
107179,SRR12754167,2020-10-01 03:57:34,70,SRX9226122,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,NextSeq 550,SRP253798,PRJNA613958


In [24]:
df_subset.sort_values(by=['LibraryLayout', 'size_MB'])

,Run,ReleaseDate,size_MB,Experiment,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,Platform,Model,SRAStudy,BioProject
72419,ERR4787607,2020-11-04 09:00:36,12,ERX4657365,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
82864,ERR4707953,2020-10-24 12:32:35,13,ERX4628379,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
42507,ERR4875827,2020-11-27 12:38:46,22,ERX4744689,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
32077,ERR4905843,2020-12-04 08:15:36,29,ERX4772667,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
82853,ERR4707958,2020-10-24 12:32:35,65,ERX4628384,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
50176,ERR4861036,2020-11-21 16:41:49,67,ERX4730419,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
107179,SRR12754167,2020-10-01 03:57:34,70,SRX9226122,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,NextSeq 550,SRP253798,PRJNA613958
80958,SRR12895638,2020-10-26 02:16:41,80,SRX9360797,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,NextSeq 550,SRP253798,PRJNA613958
91317,ERR4688199,2020-10-16 16:33:39,86,ERX4609501,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
17189,ERR5006023,2021-01-06 21:59:08,140,ERX4814955,AMPLICON,PCR,VIRAL RNA,SINGLE,ION_TORRENT,Ion Torrent PGM,ERP124690,PRJEB40971


In [25]:
df_subset.sort_values(by=['LibraryLayout', 'size_MB'], ascending=[True, False])

,Run,ReleaseDate,size_MB,Experiment,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,Platform,Model,SRAStudy,BioProject
91317,ERR4688199,2020-10-16 16:33:39,86,ERX4609501,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
80958,SRR12895638,2020-10-26 02:16:41,80,SRX9360797,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,NextSeq 550,SRP253798,PRJNA613958
107179,SRR12754167,2020-10-01 03:57:34,70,SRX9226122,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,NextSeq 550,SRP253798,PRJNA613958
50176,ERR4861036,2020-11-21 16:41:49,67,ERX4730419,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
82853,ERR4707958,2020-10-24 12:32:35,65,ERX4628384,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
32077,ERR4905843,2020-12-04 08:15:36,29,ERX4772667,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
42507,ERR4875827,2020-11-27 12:38:46,22,ERX4744689,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
82864,ERR4707953,2020-10-24 12:32:35,13,ERX4628379,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
72419,ERR4787607,2020-11-04 09:00:36,12,ERX4657365,AMPLICON,PCR,VIRAL RNA,PAIRED,ILLUMINA,Illumina NovaSeq 6000,ERP121228,PRJEB37886
17189,ERR5006023,2021-01-06 21:59:08,140,ERX4814955,AMPLICON,PCR,VIRAL RNA,SINGLE,ION_TORRENT,Ion Torrent PGM,ERP124690,PRJEB40971


---

# Split-apply-combine

Let's say we want to compute the total size of SRA runs for each `BioProject`. The strategy is:

1. **Split** the data set up according to the `'BioProject'` field
2. **Apply** a sum function to the split data sets
3. **Combine** the results into a new summary data set

This is the **split-apply-combine** strategy, put forward by Hadley Wickham in [this paper](http://dx.doi.org/10.18637/jss.v040.i01).

## Aggregation

In [26]:
grouped = df.groupby(['BioProject'])
grouped

In [27]:
df_sum = pd.DataFrame(grouped['size_MB'].sum())
df_sum.head(10)

,size_MB
BioProject,
PRJEB37513,19806
PRJEB37886,9235309
PRJEB37966,92058
PRJEB38101,533
PRJEB38351,571
PRJEB38369,1544
PRJEB38388,51684
PRJEB38459,3208
PRJEB38546,1686


In [28]:
df_sum = df_sum.reset_index()
df_sum.head()

,BioProject,size_MB
0,PRJEB37513,19806
1,PRJEB37886,9235309
2,PRJEB37966,92058
3,PRJEB38101,533
4,PRJEB38351,571


In [29]:
# Multiple columns in groupby
df.groupby(['BioProject', 'Platform']).sum(numeric_only=True).reset_index().head(10)

,BioProject,Platform,size_MB
0,PRJEB37513,ILLUMINA,19806
1,PRJEB37886,ILLUMINA,7640033
2,PRJEB37886,OXFORD_NANOPORE,1595276
3,PRJEB37966,OXFORD_NANOPORE,92058
4,PRJEB38101,ILLUMINA,533
5,PRJEB38351,ILLUMINA,571
6,PRJEB38369,ILLUMINA,1544
7,PRJEB38388,OXFORD_NANOPORE,51684
8,PRJEB38459,ILLUMINA,2672
9,PRJEB38459,OXFORD_NANOPORE,536


In [30]:
# Descriptive statistics
df.groupby(['BioProject', 'Platform'])['size_MB'].describe().head(10)

count         mean         std     min  \
BioProject Platform                                                     
PRJEB37513 ILLUMINA            244.0    81.172131   35.651535    21.0   
PRJEB37886 ILLUMINA         114178.0    66.913355   67.270998     1.0   
           OXFORD_NANOPORE   19128.0    83.400042  115.955997     1.0   
PRJEB37966 OXFORD_NANOPORE     752.0   122.417553  168.301885     5.0   
PRJEB38101 ILLUMINA              6.0    88.833333   75.058422     4.0   
PRJEB38351 ILLUMINA              6.0    95.166667   10.496031    80.0   
PRJEB38369 ILLUMINA             18.0    85.777778   24.673290    49.0   
PRJEB38388 OXFORD_NANOPORE     583.0    88.651801   65.441487     1.0   
PRJEB38459 ILLUMINA              1.0  2672.000000         NaN  2672.0   
           OXFORD_NANOPORE       1.0   536.000000         NaN   536.0   

                                25%     50%      75%     max  
BioProject Platform                                           
PRJEB37513 ILLUMINA           36.00    92.5   108.00   161.0  
PRJEB37886 ILLUMINA           22.00    59.0    89.00  3536.0  
           OXFORD_NANOPORE    19.00    51.0   108.00  3845.0  
PRJEB37966 OXFORD_NANOPORE    67.00   110.0   150.00  2846.0  
PRJEB38101 ILLUMINA           25.00    89.0   147.00   181.0  
PRJEB38351 ILLUMINA           88.75    96.0   103.25   107.0  
PRJEB38369 ILLUMINA           61.25    92.5   104.00   124.0  
PRJEB38388 OXFORD_NANOPORE    44.00    75.0   122.00   421.0  
PRJEB38459 ILLUMINA         2672.00  2672.0  2672.00  2672.0  
           OXFORD_NANOPORE   536.00   536.0   536.00   536.0

In [31]:
# Custom aggregations
df.groupby(['BioProject', 'Platform']).agg({'size_MB': np.mean, 'Run': 'nunique'}).head(10)

/var/folders/f0/hrhg01rs28sgr9r5v8k3rwrw0000gn/T/ipykernel_10117/3276325559.py:2: FutureWarning: The provided callable <function mean at 0x10846a160> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  df.groupby(['BioProject', 'Platform']).agg({'size_MB': np.mean, 'Run': 'nunique'}).head(10)


size_MB     Run
BioProject Platform                            
PRJEB37513 ILLUMINA           81.172131     244
PRJEB37886 ILLUMINA           66.913355  114178
           OXFORD_NANOPORE    83.400042   19128
PRJEB37966 OXFORD_NANOPORE   122.417553     752
PRJEB38101 ILLUMINA           88.833333       6
PRJEB38351 ILLUMINA           95.166667       6
PRJEB38369 ILLUMINA           85.777778      18
PRJEB38388 OXFORD_NANOPORE    88.651801     583
PRJEB38459 ILLUMINA         2672.000000       1
           OXFORD_NANOPORE   536.000000       1

---

## Tidying a data set with melt

The most useful function for tidying data is `pd.melt()`. Let's demonstrate with a coverage dataset:

In [32]:
# Download coverage data if needed
import os
coverage_file = "coverage.tsv.gz"
if not os.path.exists(coverage_file):
    from urllib.request import urlretrieve
    urlretrieve("https://zenodo.org/records/10680470/files/coverage.tsv.gz", coverage_file)
    print(f"Downloaded: {coverage_file}")
else:
    print(f"Using pre-loaded file: {coverage_file}")

df_cov = pd.read_csv(coverage_file, sep='\t')
df_cov.head()

Using pre-loaded file: coverage.tsv.gz


,#chr,start,end,SRR12733539.bam,SRR12733570.bam,SRR12733581.bam,SRR12733607.bam,SRR12733616.bam,SRR12733619.bam
0,NC_045512.2,0,100,259.0,3490.0,1645.0,399.0,598.0,107.0
1,NC_045512.2,100,200,720.0,9106.0,3780.0,1063.0,1404.0,272.0
2,NC_045512.2,200,300,979.0,14207.0,5054.0,1450.0,1921.0,344.0
3,NC_045512.2,300,400,983.0,15438.0,5212.0,1396.0,1908.0,328.0
4,NC_045512.2,400,500,1297.0,17257.0,6174.0,1567.0,2021.0,385.0


These data are not tidy. When we melt the data frame, the data within it (called **values**) become a single column. The headers (called **variables**) also become new columns.

![Dataframe melt](https://pandas.pydata.org/docs/_images/07_melt.svg)

In [33]:
melted = pd.melt(df_cov, 
                 value_name='coverage', 
                 var_name='sample',
                 value_vars=df_cov.columns[3:],
                 id_vars=['start', 'end'])

melted.head()

,start,end,sample,coverage
0,0,100,SRR12733539.bam,259.0
1,100,200,SRR12733539.bam,720.0
2,200,300,SRR12733539.bam,979.0
3,300,400,SRR12733539.bam,983.0
4,400,500,SRR12733539.bam,1297.0


In [34]:
melted.groupby(['sample'])['coverage'].describe()

,count,mean,std,min,25%,50%,75%,max
sample,,,,,,,,
SRR12733539.bam,100.0,1271.75,295.525499,259.0,1093.00,1310.0,1484.25,1963.0
SRR12733570.bam,100.0,10112.04,3412.618417,1843.0,8582.25,10615.5,12062.75,18673.0
SRR12733581.bam,100.0,5800.16,1360.497731,1645.0,5038.50,6118.5,6766.50,8100.0
SRR12733607.bam,100.0,1509.15,389.486359,399.0,1334.50,1571.0,1834.75,2159.0
SRR12733616.bam,100.0,1691.37,444.477395,414.0,1446.50,1820.0,2030.00,2366.0
SRR12733619.bam,100.0,359.54,77.393318,107.0,321.50,384.5,411.00,482.0


To get back from melted (narrow) format to wide format, use `pivot()`:

![Dataframe pivot](https://pandas.pydata.org/docs/_images/07_pivot.svg)

In [35]:
melted.pivot(index=['start', 'end'], columns='sample', values='coverage').head()

,sample,SRR12733539.bam,SRR12733570.bam,SRR12733581.bam,SRR12733607.bam,SRR12733616.bam,SRR12733619.bam
start,end,,,,,,
0,100,259.0,3490.0,1645.0,399.0,598.0,107.0
100,200,720.0,9106.0,3780.0,1063.0,1404.0,272.0
200,300,979.0,14207.0,5054.0,1450.0,1921.0,344.0
300,400,983.0,15438.0,5212.0,1396.0,1908.0,328.0
400,500,1297.0,17257.0,6174.0,1567.0,2021.0,385.0


---

# Working with multiple tables

Working with multiple tables often involves joining them on a common key.

![Left join](https://pandas.pydata.org/docs/_images/08_merge_left.svg)

In [36]:
df1 = pd.DataFrame({"key": ["A", "B", "C", "D"], "value": np.random.randn(4)})
df2 = pd.DataFrame({"key": ["B", "D", "D", "E"], "value": np.random.randn(4)})

In [37]:
df1

,key,value
0,A,0.934504
1,B,0.294242
2,C,-0.326532
3,D,-0.617024


In [38]:
df2

,key,value
0,B,-1.078890
1,D,1.230878
2,D,1.186422
3,E,1.174122


## Inner join

![Inner join](https://upload.wikimedia.org/wikipedia/commons/thumb/1/18/SQL_Join_-_07_A_Inner_Join_B.svg/234px-SQL_Join_-_07_A_Inner_Join_B.svg.png)

In [39]:
pd.merge(df1, df2, on="key")

,key,value_x,value_y
0,B,0.294242,-1.078890
1,D,-0.617024,1.230878
2,D,-0.617024,1.186422


## Left join

![Left join](https://upload.wikimedia.org/wikipedia/commons/thumb/d/dc/SQL_Join_-_01b_A_Left_Join_B.svg/234px-SQL_Join_-_01b_A_Left_Join_B.svg.png)

In [40]:
pd.merge(df1, df2, on="key", how="left").fillna('.')

,key,value_x,value_y
0,A,0.934504,.
1,B,0.294242,-1.07889
2,C,-0.326532,.
3,D,-0.617024,1.230878
4,D,-0.617024,1.186422


## Right join

![Right join](https://upload.wikimedia.org/wikipedia/commons/thumb/5/5f/SQL_Join_-_03_A_Right_Join_B.svg/234px-SQL_Join_-_03_A_Right_Join_B.svg.png)

In [41]:
pd.merge(df1, df2, on="key", how="right").fillna('.')

,key,value_x,value_y
0,B,0.294242,-1.078890
1,D,-0.617024,1.230878
2,D,-0.617024,1.186422
3,E,.,1.174122


## Full (outer) join

![Full join](https://upload.wikimedia.org/wikipedia/commons/thumb/6/61/SQL_Join_-_05_A_Full_Join_B.svg/234px-SQL_Join_-_05_A_Full_Join_B.svg.png)

In [42]:
pd.merge(df1, df2, on="key", how="outer").fillna('.')

,key,value_x,value_y
0,A,0.934504,.
1,B,0.294242,-1.07889
2,C,-0.326532,.
3,D,-0.617024,1.230878
4,D,-0.617024,1.186422
5,E,.,1.174122


---

# Putting it all together: Pandas + Altair

## Understanding [Altair](https://altair-viz.github.io/)

Vega-Altair is a declarative statistical visualization library for Python. It offers a powerful and concise grammar that enables you to quickly build a wide range of statistical visualizations.

In [43]:
import pandas as pd
import altair as alt
from datetime import date
today = date.today()

In [44]:
# Download ENA data if needed
import os
ena_file = "ena.tsv.gz"
if not os.path.exists(ena_file):
    from urllib.request import urlretrieve
    urlretrieve("https://zenodo.org/records/10680776/files/ena.tsv.gz", ena_file)
    print(f"Downloaded: {ena_file}")
else:
    print(f"Using pre-loaded file: {ena_file}")

# Read a larger dataset
sra = pd.read_csv(
    ena_file,
    compression='gzip',
    sep="\t",
    low_memory=False,
    nrows=100000  # Limit rows for faster loading
)

Using pre-loaded file: ena.tsv.gz


In [45]:
len(sra)

100000

In [46]:
sra.sample(5)

,study_accession,base_count,accession,collection_date,country,culture_collection,description,sample_collection,sample_title,sequencing_method,...,library_name,library_construction_protocol,library_layout,instrument_model,instrument_platform,isolation_source,isolate,investigation_type,collection_date_submitted,center_name
41695,PRJNA631061,4.546184e+07,SAMN16796711,2020-03-15,USA: Minnesota,NaN,Illumina MiSeq sequencing; SARS-CoV-2 patient ...,NaN,This sample has been submitted by pda|kaganr o...,NaN,...,MN-QDX-985,NaN,PAIRED,Illumina MiSeq,ILLUMINA,NaN,patient,NaN,2020-03-15,SUB8553683
93673,PRJEB37886,5.397910e+08,SAMEA12285660,2022-01-05,United Kingdom,NaN,Illumina NovaSeq 6000 sequencing; Illumina Nov...,NaN,COG-UK/MILK-31327AC,NaN,...,NT1717062I / HT-128819:E10,NaN,PAIRED,Illumina NovaSeq 6000,ILLUMINA,NaN,not provided,NaN,2022-01-05,SC
1924,PRJNA731152,2.413069e+09,SAMN25060377,2022-01-03,USA: California,NaN,Illumina NovaSeq 6000 sequencing,NaN,CDC Sars CoV2 Sequencing Baseline Constellation,NaN,...,CDC Flu SC2,Fulgent COVIDSeq v4,PAIRED,Illumina NovaSeq 6000,ILLUMINA,nasal swab,SARS-CoV-2/Human/USA/CA-CDC-FG-232953/2022,NaN,2022-01-03,NaN
29031,PRJEB37886,6.345591e+08,SAMEA12519121,2022-01-10,United Kingdom,NaN,Illumina NovaSeq 6000 sequencing; Illumina Nov...,NaN,COG-UK/MILK-31F8599,NaN,...,NT1718906I / HT-129334:F2,NaN,PAIRED,Illumina NovaSeq 6000,ILLUMINA,NaN,not provided,NaN,2022-01-10,SC
3511,PRJNA716984,6.774141e+06,SAMN32624846,2022-12-27,USA: California,NaN,Sequel II sequencing,NaN,CDC Sars CoV2 Sequencing Baseline Constellation,NaN,...,Unknown,Freed primers,PAIRED,Sequel II,PACBIO_SMRT,Nasal Swabs,SARS-CoV-2/Human/USA/CA-CDC-LC0974026/2022,NaN,2022-12-27,NaN


## Cleaning the data

In [47]:
# Convert collection_date to datetime
# Note: errors='coerce' converts unparseable dates to NaT (Not a Time)
# This is acceptable here because we will filter out invalid dates in the next step
sra = sra.assign(collection_date=pd.to_datetime(sra["collection_date"], errors='coerce'))

In [48]:
print('Earliest entry:', sra['collection_date'].min())
print('Latest entry:', sra['collection_date'].max())

Earliest entry: 2019-12-30 00:00:00
Latest entry: 2023-01-20 00:00:00


::: {.callout-warning}
## Data Quality

Don't get surprised here - the metadata is only as good as the person who entered it. So, **when you enter metadata for your sequencing data -- pay attention!!!**
:::

In [49]:
# Filter to valid date range using explicit Timestamp objects for clarity
sra = sra[
    (sra['collection_date'] >= pd.Timestamp('2020-01-01')) 
    & 
    (sra['collection_date'] <= pd.Timestamp('2023-12-31'))
]

In [50]:
# Aggregate data for heatmap
heatmap_2d = sra.groupby(
    ['instrument_platform', 'library_strategy']
).agg(
    {'run_accession': 'nunique'}
).reset_index()

heatmap_2d

,instrument_platform,library_strategy,run_accession
0,BGISEQ,AMPLICON,1
1,BGISEQ,OTHER,13
2,BGISEQ,RNA-Seq,2
3,BGISEQ,Targeted-Capture,2
4,DNBSEQ,AMPLICON,3
5,ILLUMINA,AMPLICON,78262
6,ILLUMINA,OTHER,2
7,ILLUMINA,RNA-Seq,524
8,ILLUMINA,Targeted-Capture,272
9,ILLUMINA,WCS,2


## Plotting the data

In [51]:
back = alt.Chart(heatmap_2d).mark_rect(opacity=1).encode(
    x=alt.X(
        "instrument_platform:N",
        title="Instrument"
    ),
    y=alt.Y(
        "library_strategy:N",
        title="Strategy",
        axis=alt.Axis(orient='right')
    ),
    color=alt.Color(
        "run_accession:Q",
        title="# Samples",
        scale=alt.Scale(
            scheme="goldred",
            type="log"
        ),
    ),
    tooltip=[
        alt.Tooltip(
            "instrument_platform:N",
            title="Machine"
        ),
        alt.Tooltip(
            "run_accession:Q",
            title="Number of runs"
        ),
        alt.Tooltip(
            "library_strategy:N",
            title="Protocol"
        )
    ]
).properties(
    width=500,
    height=150,
    title={
        "text": ["Breakdown of datasets from ENA",
                 "by Platform and Library Strategy"],
        "subtitle": "(Sample of 100k records)"
    }
)

back

alt.Chart(...)

In [52]:
# Add text labels
front = back.mark_text(
    align="center",
    baseline="middle",
    fontSize=12,
    fontWeight="bold",
).encode(
    text=alt.Text("run_accession:Q", format=",.0f"),
    color=alt.condition(
        alt.datum.run_accession > 200,
        alt.value("white"),
        alt.value("black")
    )
)

# Combine layers
back + front

alt.LayerChart(...)

## Summary

In this lecture, we covered:

1. **Pandas basics**: DataFrames, indexing with `loc` and `iloc`
2. **Boolean indexing**: Filtering data with conditions
3. **Calculations**: Vectorized operations on columns
4. **Tidy data**: Principles of data organization
5. **Split-apply-combine**: Using `groupby()` for aggregations
6. **Reshaping**: `melt()` and `pivot()` for transforming data
7. **Joins**: Combining tables with `merge()`
8. **Visualization**: Creating plots with Altair

These skills form the foundation of data analysis in Python!